In [ ]:
import os
from pathlib import Path

from q_rewrite.clients import ModelClient
from q_rewrite.optimizers import Optimizer
from q_rewrite.utilities.logging import get_logger
from q_rewrite.utilities.os import load_env_file
from q_rewrite.verifiers import QiskitVerifier


# load the env vars
load_env_file(project_root_path=Path(__file__).parent)
logger = get_logger()
model_client = ModelClient(
    api_key=os.environ["MODEL_API_KEY"],
    base_url=os.environ["MODEL_API_BASE_URL"],
    logger=logger,
    model=os.environ["MODEL"],
)
optimizer = Optimizer(
    model_client=model_client,
    logger=logger,
    verifier=QiskitVerifier(logger=logger),
)

## 1. Simple Cancellation

```qasm
OPENQASM 3;
include "stdgates.inc";

qubit[2] q;

h q[0];
cx q[0], q[1];

x q[0];
x q[0];

cx q[0], q[1];
```

**Optimizations**:
- Remove inverse $X$ gates because $X^{2} = I$.

In [ ]:
qasm = Path("./examples/01_cancellation.qasm").read_text(encoding="utf-8")
optimizer.optimize(circuit=qasm)

## 2. Non-syntactically Adjacent Gates

```qasm
OPENQASM 3;
include "stdgates.inc";

qubit[2] q;

x q[0];
h q[1];
x q[0];
```

**Optimizations**:
- Remove inverse $X$ gates on $q_0$ because $X^{2} = I$.

> **NOTE:** The $X$ gates on the first qubit are semantically adjacent, are not syntactically adjacent.

In [ ]:
qasm = Path("./examples/02_non_syntactically_adjacent.qasm").read_text(encoding="utf-8")
optimizer.optimize(circuit=qasm)

## 3. Bell Redundancy

```qasm
OPENQASM 3;
include "stdgates.inc";

qubit[2] q;

h q[0];
x q[0];
x q[0];

cx q[0], q[1];

rz(0.25) q[1];
rz(-0.25) q[1];
```

**Optimizations**:
- Remove inverse $X$ gates on $q_0$ because $X^{2} = I$.
- Remove the $R_{Z}$ rotations on $q_1$ since $R_{Z}(0.25)R_{Z}(-0.25) = I$.

In [ ]:
qasm = Path("./examples/03_bell_redundant.qasm").read_text(encoding="utf-8")
optimizer.optimize(circuit=qasm)

## 4. GHZ (3-qubit)

```qasm
OPENQASM 3;
include "stdgates.inc";

qubit[3] q;

h q[0];
cx q[0], q[1];
cx q[1], q[2];

z q[2];
z q[2];
```

**Optimizations**:
- Remove inverse $Z$ gates on $q_2$ because $Z^{2} = I$.

In [ ]:
qasm = Path("./examples/04_ghz3.qasm").read_text(encoding="utf-8")
optimizer.optimize(circuit=qasm)

## 5. GHZ (4-qubit)

```qasm
OPENQASM 3;
include "stdgates.inc";

qubit[4] q;

h q[0];

cx q[0], q[1];
cx q[1], q[2];
cx q[2], q[3];

x q[1];
x q[1];

h q[3];
h q[3];

cx q[0], q[1];
cx q[0], q[1];
```

**Optimizations**:
- Remove inverse $X$ gates on $q_1$ because $X^{2} = I$.
- Remove inverse $H$ gates on $q_3$ because $H^{2} = I$.
- Remove inverse $CNOT$ gates on $q_0$ and $q_1$ because $CNOT$ is self-inverse.

In [ ]:
qasm = Path("./examples/05_ghz4_redundant.qasm").read_text(encoding="utf-8")
optimizer.optimize(circuit=qasm)

## 6. QAOA (MaxCut)

```qasm
OPENQASM 3;
include "stdgates.inc";

qubit[3] q;

h q[0];
h q[1];
h q[2];

cx q[0], q[1];
rz(0.7) q[1];
cx q[0], q[1];

cx q[1], q[2];
rz(0.7) q[2];
cx q[1], q[2];

cx q[0], q[2];
rz(0.7) q[2];
cx q[0], q[2];

rx(1.1) q[0];
rx(1.1) q[1];
rx(1.1) q[2];

rz(0.4) q[0];
rz(-0.4) q[0];
```

**Optimizations**:
- Each $CNOT$/$R_{Z}$ block is a conjugation by $CNOT$, and since $R_{Z}(\theta)$ is diagonal, so they can be rewritten as controlled $R_{Z}$ operations:
$$
  CNOT(control, target)(I_{control} \otimes R_{Z}(\theta)_{t})CNOT(control, target) = CRZ_(control, target)(\theta)
$$
> **NOTE:** A controlled $R_{Z}$ may not be a cheaper hardware-aware operation than a $CNOT$/$R_{X}$ block.
- Remove the $R_{Z}$ rotations on $q_0$ since $R_{Z}(0.4)R_{Z}(-0.4) = I$.

In [ ]:
qasm = Path("./examples/06_qaoa_maxcut3.qasm").read_text(encoding="utf-8")
optimizer.optimize(circuit=qasm)

## 7. QFT (3-qubit)

```qasm
OPENQASM 3;
include "stdgates.inc";

qubit[3] q;

h q[0];
cp(pi / 2) q[1], q[0];
cp(pi / 4) q[2], q[0];

h q[1];
cp(pi / 2) q[2], q[1];

h q[2];

swap q[0], q[2];

rz(0.3) q[1];
rz(-0.3) q[1];
```

**Optimizations**:
- Merge $RZ$ rotations on $q_1$ since $R_{Z}(0.3)R_{Z}(-0.3) = I$.

> **NOTE:** Since a $SWAP$ gate weighs a significant cost, the final $SWAP$ gate can be removed if the output bit-reversal is handled by classical interpretation or by mapping logical qubits to reversed physical positions. It will be interesting to see if the model catches this!

In [ ]:
qasm = Path("./examples/07_qft3_redundant.qasm").read_text(encoding="utf-8")
optimizer.optimize(circuit=qasm)

## 8. QAOA (MaxCut) - back with a vengence!

```qasm
OPENQASM 3;
include "stdgates.inc";

qubit[3] q;

h q[0];
h q[1];
h q[2];

cx q[0], q[1];
rz(0.6) q[1];
cx q[0], q[1];

cx q[1], q[2];
rz(0.6) q[2];
cx q[1], q[2];

cx q[0], q[2];
rz(0.6) q[2];
cx q[0], q[2];

rx(1.0) q[0];
rx(1.0) q[1];
rx(1.0) q[2];

cx q[0], q[1];
rz(0.3) q[1];
cx q[0], q[1];

cx q[1], q[2];
rz(0.3) q[2];
cx q[1], q[2];

cx q[0], q[2];
rz(0.3) q[2];
cx q[0], q[2];

rx(0.8) q[0];
rx(0.8) q[1];
rx(0.8) q[2];

h q[0];
h q[0];
```

**Optimizations**:
- Merge the $R_{X}$ rotation blocks on each qubit into one gate since $R_{X}(1.0)R_{X}(0.8) = R_{X}(1.8)$.
> **NOTE:** Since the intervening operations in-between these blocks do not matter because they act on disjoint qubits.
- Like before in [6. QAOA (MaxCut)](#6-qaoa-maxcut), the $CNOT$/$R_{Z}$ blocks can be rewritten as controlled $R_{Z}$ operations.
- Remove inverse $H$ gates on $q_0$ because $H^{2} = I$.

In [ ]:
qasm = Path("./examples/08_qaoa_maxcut3_p2.qasm").read_text(encoding="utf-8")
optimizer.optimize(circuit=qasm)

## 9. Teleportation

```qasm
OPENQASM 3;
include "stdgates.inc";

qubit[3] q;
bit[2] c;

x q[0];
h q[1];
cx q[1], q[2];
h q[1];
cx q[0], q[1];
h q[0];

c[0] = measure q[0];
c[1] = measure q[1];

if (c[1]) {
    x q[2];
}

if (c[0]) {
    z q[2];
}
```

**Optimizations**:
- In the middle sequence of the first block, there contains a substitution for the $H(q_{1})CNOT(q_{1}, q_{2})H(q_{1})$ into a controlled-$Z$ gate since:
$$
  H(control) CNOT(control, target) H(control) = CZ(control, target)
$$
> **NOTE:** A controlled-$Z$ may not be a cheaper hardware-aware operation than a $H(control) CNOT(control, target) H(control)$ block.

In [ ]:
qasm = Path("./examples/09_teleportation.qasm").read_text(encoding="utf-8")
optimizer.optimize(circuit=qasm)